# Dataload - Modelagem de Dados

### Importando as bibliotecas necessárias

In [11]:
from faker import Faker # para geração de dados ficticios
import random # para gerar numeros aleatorios
from datetime import datetime # para gerar os valores de data da tabela informações
import pandas as pd
from dotenv import load_dotenv
import sqlalchemy as sqla
import psycopg2
import os
import bcrypt as bcr
from passlib.hash import bcrypt
import re
from datetime import timedelta
import psycopg2
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()



# definindo lingua do faker para portugues 
# e a seed com o mesmo numero em ambas
# para a geração do mesmo padrão de valores
fake = Faker("pt-BR")
Faker.seed()
random.seed(1)

### Função para tratar os dados criados pelo Faker

### Código para criar os insert que vai no documento data_load.sql

In [12]:
qt_motorista=0
insert_cliente = "\nINSERT INTO Cliente(nome_cliente, cpf, telefone, email, score, id_endereco) VALUES\n"
insert_conta= "\nINSERT INTO Conta( numero_conta , senha ,  tipo_conta ,  saldo ,  data_abertura, status ,id_cliente) VALUES\n"
insert_cartao = "\nINSERT INTO Cartao(numero_cartao , senha , validade , cvv, tipo_cartao ,status, id_conta, limite_cartao) VALUES\n"
insert_transacao = "\nINSERT INTO Transacao( tipo_transacao , valor,  data_transacao , descricao,  id_conta, id_cartao, status) VALUES\n"
insert_emprestimo  = "\nINSERT INTO Emprestimo( valor_total , taxa_juros,  quantidade_parcelas ,  data_contratacao,  status , id_conta ) VALUES\n"
insert_endereco = "\nINSERT INTO Endereco(cep, rua , numero ,  complemento,  bairro, cidade,  estado) VALUES \n"



idEndereco=1
conta_acumulada = 0
cartao_acumulado = 0




for id_cliente in range(1,501):
    
    nome_completo = f"{fake.first_name()} {fake.last_name()} {fake.last_name()}"
    
    # (cep, rua , numero ,  complemento,  bairro, cidade,  estado)
    insert_endereco += f"('{fake.postcode(formatted=False)}', '{fake.street_name()}', '{random.randint(1, 9999)}', '{fake.sentence()}', '{fake.neighborhood().replace("'",'')}' , '{fake.city()}', '{fake.state_abbr()}'),\n"
    
    # (nome_cliente, cpf, telefone, email, score, id_endereco)
    insert_cliente += f"('{nome_completo}', '{fake.cpf().replace('.', '').replace('-', '')}', '{fake.bothify(text='##9########')}', '{fake.company_email()}', {round(random.uniform(0, 1000), 2)}, {idEndereco}),\n"
    
    idEndereco += 1
    
    for id_conta in range(conta_acumulada+1,random.randint(conta_acumulada+2,conta_acumulada+4)):
                
        data_abertura_obj = fake.date_time_between(start_date='-5y', end_date='now')
        data_abertura_sql = data_abertura_obj.strftime('%Y-%m-%d %H:%M:%S')
        tipo_conta = fake.random_element(elements=["UNDERAGE", "BASIC", "BLACK"])
        
        # ( numero_conta , senha ,  tipo_conta ,  saldo ,  data_abertura, status ,id_cliente)
        insert_conta+= f"( '{fake.random_number(digits=12, fix_len=True)}' , '{fake.password()}', '{tipo_conta}' ,  {round(random.uniform(10.00, 15000.00), 2)} , '{data_abertura_sql}' , '{fake.random_element(elements=["ATIVA", "INATIVA", "BLOQUEADA"])}', {id_cliente}),\n"
    
        for id_cartao in range(cartao_acumulado+1,random.randint(cartao_acumulado+2,cartao_acumulado+4)):
            
            data_v = fake.date_between(start_date=data_abertura_obj, end_date='+5y')
            
            hoje = datetime.now().date()

            status = ("INATIVO" if random.random() <= 0.9 else fake.random_element(elements=["ATIVO", "BLOQUEADO"])) if data_v < hoje else fake.random_element(elements=["ATIVO", "INATIVO", "BLOQUEADO"])
            limite = round(random.uniform(5000, 50000), 2) if tipo_conta== 'BLACK' else (round(random.uniform(2000, 15000), 2) if tipo_conta == 'MULTIPLO' else round(random.uniform(1000, 5000), 2))
            tipo_cartao = fake.random_element(elements=['DEBITO', 'CREDITO', 'MULTIPLO'])
            
            # (numero_cartao , senha , validade , cvv, tipo_cartao ,status, id_conta, limite_cartao)
            insert_cartao += f"('{fake.random_number(digits=16, fix_len=True)}', '{fake.password()}' , '{data_v}' , '{fake.random_number(digits=3, fix_len=True)}', '{tipo_cartao}' , '{status}' , {id_conta}, {limite} ),\n"
            
            for id_transacao in range(1,random.randint(2,16)):
                
                data_transacao = fake.date_time_between(start_date=data_abertura_obj, end_date='now').strftime('%Y-%m-%d %H:%M:%S')
                if tipo_cartao == 'MULTIPLO':
                    tipo_transacao = fake.random_element(elements=['DEBITO', 'CREDITO'])
                else:
                    tipo_transacao = tipo_cartao
                
                
                # ( tipo_transacao , valor,  data_transacao , descricao,  id_conta, id_cartao, status)
                insert_transacao += f"( '{tipo_transacao}' , {round(random.uniform(5.00, 500.00), 2)},  '{data_transacao}', '{fake.sentence()}',  {id_conta}, {id_cartao}, '{fake.random_element(elements=['VALIDO', 'INVALIDO'])}'),\n"

            cartao_acumulado +=1

        for id_transacao in range(1,random.randint(2,16)):
                
                data_transacao = fake.date_time_between(start_date=data_abertura_obj, end_date='now').strftime('%Y-%m-%d %H:%M:%S')
                tipo_transacao = fake.random_element(elements=['PIX', 'TED', 'BOLETO'])
                
                # ( tipo_transacao , valor,  data_transacao , descricao,  id_conta, id_cartao, status)
                insert_transacao += f"( '{tipo_transacao}' , {round(random.uniform(5.00, 500.00), 2)},  '{data_transacao}', '{fake.sentence()}',  {id_conta}, 'NULL', '{fake.random_element(elements=['VALIDO', 'INVALIDO'])}'),\n"

        for id_emprestimo in  range(1,random.randint(2,4)):
            
            # ( valor_total , taxa_juros,  quantidade_parcelas ,  data_contratacao,  status , id_conta )
            insert_emprestimo += f"({round(random.uniform(1000.00, 50000.00), 2)} , {round(random.uniform(1.5, 7.9), 2) },  {random.choice([12, 24, 36, 48, 60, 72])} ,  '{fake.date_time_between(start_date=data_abertura_obj, end_date='now').strftime('%Y-%m-%d %H:%M:%S')}',  '{fake.random_element(elements=['ATIVO', 'QUITADO', 'PENDENTE', 'CANCELADO'])}' , {id_conta} ),\n"


        conta_acumulada += 1
 
lista_inserts = [insert_endereco,insert_cliente,insert_conta,insert_cartao,insert_transacao,insert_emprestimo]       
        
        

#### Salvar os inserts em um documento

In [13]:
with open("sql/data_load.sql", "w", encoding='utf-8') as o:
    o.write("-- Script Dataload")  

for insert in lista_inserts:
    insert = insert[:len(insert)-2]
    insert+=";\n\n\n"
    insertCorrigido = insert.replace("'NULL'","NULL")
    try:
        # with open("sql/mvp.sql", "a", encoding='utf-8') as o:
        #     o.write(insertCorrigido)
        with open("sql/data_load.sql", "a", encoding='utf-8') as o:
            o.write(insertCorrigido)
    except FileNotFoundError:
        print('Arquivo não encontado ao escrever o insert')

#### Fazer o insert diretamente no banco


In [14]:

db_name = os.getenv("DB_NAME")
db_user = os.getenv("DB_USER")
db_pass = os.getenv("DB_PASS")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")

try:
    conn = psycopg2.connect(
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASS"),
        host=os.getenv("DB_HOST"),
        port=os.getenv("DB_PORT")
    )

    cur = conn.cursor()
    total_adicionado = 0
    
    for script in lista_inserts:
        
        script = script[:len(script)-2]
        script+=";"
        script = script.replace("'NULL'","NULL")
        cur.execute(script.strip())
        
        total_adicionado += cur.rowcount 
        
    conn.commit() 
    
    print(f"Sucesso! Total de registros adicionados: {total_adicionado}")
    cur.close()
except Exception as e:
    print(f"Erro ao conectar ou ler dados: {e}")

finally:
    if 'conn' in locals():
        conn.close()
        print("Conexão fechada.")


Sucesso! Total de registros adicionados: 29045
Conexão fechada.
